In [1]:
from azure.ai.ml import MLClient
from azure.ai.ml.entities import ManagedOnlineEndpoint, ManagedOnlineDeployment
from azure.identity import DefaultAzureCredential

SUBSCRIPTION_ID = "126ac41d-e5e6-4702-b949-9d141a1b7042"
RESOURCE_GROUP = "rg-mlab1-ali886"
WORKSPACE = "mlw-lab1-ali886"

ml_client = MLClient(DefaultAzureCredential(), SUBSCRIPTION_ID,
                     RESOURCE_GROUP, WORKSPACE)
print("Connected:", ml_client.workspace_name)

Connected: mlw-lab1-ali886


In [2]:
try:
    ml_client.online_endpoints.begin_delete(name="knn-maint-ali886").result()
    print("Old endpoint deleted")
except Exception as e:
    print("Delete message:", e)

........................................

In [ ]:
ENDPOINT_NAME = "knn-maint-ali886"

endpoint = ManagedOnlineEndpoint(
    name=ENDPOINT_NAME,
    description="KNN predictive maintenance (notebook model)",
    auth_mode="key",
)
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print("Endpoint created:", ENDPOINT_NAME)

In [ ]:
model = ml_client.models.get(name="knn-maintenance-notebook", label="latest")
print("Deploying model:", model.name, "version", model.version)

deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=ENDPOINT_NAME,
    model=model,
    instance_type="Standard_DS2_v2",
    instance_count=1,
)
ml_client.online_deployments.begin_create_or_update(deployment).result()

endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print("Deployment ready")